# DoRA-PVS Challenge Replication (VICOROBIGR Winning Method)
### Free GPU Training on Google Colab / Kaggle

This notebook runs the complete **synthetic generator**, **3D DynUNet training**, and **evaluation** using a free cloud GPU (T4 / P100 / V100).

## Step 1: Check GPU & Clone / Upload Project

In [ ]:
!nvidia-smi

## Step 2: Install Core Dependencies

In [ ]:
!pip install -q monai nibabel scikit-image scikit-learn scipy matplotlib nilearn timm

## Step 3: Build Anatomical Atlas Canvases

In [ ]:
# Build MNI-derived anatomical canvases first. This avoids the old toy ellipse fallback.
!python src/generator/build_atlases.py --num_atlases 10 --output_dir data/atlases

## Step 4: Generate Atlas-Based Synthetic MRI Volumes

In [ ]:
# Run generator to create 30 atlas-based, multi-contrast scans
!python src/generator/generate_dataset.py --num_samples 30 --output_dir data/synthetic_real
!python src/generator/inspect_data.py --data_dir data/synthetic_real --output_image data/inspection_report_real.png

## Step 5: Train 3D Multi-Class DynUNet

In [ ]:
!python src/models/train.py \
    --data_dir data/synthetic_real \
    --checkpoint_dir checkpoints \
    --epochs 30 \
    --batch_size 2 \
    --patch_size 96 \
    --learning_rate 0.001

## Step 6: Evaluate Model on Challenge Metrics (AUPRC, clDice, Lesion-wise DSC)

In [ ]:
!python src/evaluation/evaluate.py \
    --data_dir data/synthetic_real \
    --checkpoint checkpoints/best_dynunet_multiclass.pth

## Step 7: Download Your Trained Weights (`.pth`) Back to Local Machine

In [ ]:
from google.colab import files
files.download('checkpoints/best_dynunet_multiclass.pth')